# Superstore Sales & Profitability Analysis with AI-Based Prediction

**IBM SkillsBuild Data Analytics with AI — Academic Internship**

---

| | |
|---|---|
| **Student** | Mahammadsakib Mulla |
| **Programme** | IBM SkillsBuild Data Analytics with AI |
| **Dataset** | Sample - Superstore (US Retail, 2014–2017) |
| **Records** | 9,994 transactions |
| **Objective** | EDA, Business KPI Analysis, and AI-based Sales & Profitability Prediction |

---

## Table of Contents
1. [Introduction & Problem Statement](#intro)
2. [Technology Stack](#tech)
3. [Data Loading](#load)
4. [Data Inspection](#inspect)
5. [Data Cleaning & Quality Checks](#clean)
6. [Exploratory Data Analysis](#eda)
7. [Business KPIs & Analysis](#kpi)
8. [Feature Engineering](#features)
9. [Sales Prediction — Regression Models](#regression)
10. [Profitability Classification Models](#classification)
11. [Model Evaluation & Comparison](#eval)
12. [Feature Importance & Interpretability](#importance)
13. [Business Insights & Recommendations](#insights)
14. [Conclusion](#conclusion)

---
<a id='intro'></a>
## 1. Introduction & Problem Statement

### Introduction
This project analyses four years (2014–2017) of transaction data from a US-based retail chain — the **Superstore** dataset — which covers over 9,900 order line items across Furniture, Office Supplies, and Technology product categories.

### Problem Statement
The business faces two critical challenges:
1. **~19.4% of all transactions result in a loss** — the business needs to understand the drivers of these losses and be able to predict them before an order ships.
2. **Sales are unevenly distributed** — a small number of high-value products and regions account for the majority of revenue. Predictive insights can guide sales strategy.

### Objectives
- Perform comprehensive Exploratory Data Analysis (EDA)
- Calculate business KPIs (Total Sales, Profit, Margin, etc.)
- Build a **Sales Prediction** model (regression)
- Build a **Profitability Classification** model (predict profit vs loss)
- Derive actionable business recommendations from the analysis

---
<a id='tech'></a>
## 2. Technology Stack

| Library | Purpose |
|---|---|
| **pandas** | Data loading, manipulation, aggregation |
| **numpy** | Numerical operations |
| **matplotlib / seaborn** | Static visualizations |
| **scikit-learn** | Machine learning models, preprocessing, evaluation |
| **joblib** | Model serialization |
| **FastAPI** | REST API backend |
| **Streamlit** | Interactive web dashboard |

---
<a id='load'></a>
## 3. Data Loading

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import joblib
import os

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.ensemble import (RandomForestRegressor, RandomForestClassifier,
                               GradientBoostingRegressor, GradientBoostingClassifier)
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             classification_report, confusion_matrix, roc_auc_score,
                             ConfusionMatrixDisplay)

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

RANDOM_STATE = 42
CHARTS_DIR = '../outputs/charts'
os.makedirs(CHARTS_DIR, exist_ok=True)

print('Libraries loaded.')
print(f'pandas {pd.__version__}  |  numpy {np.__version__}')

In [ ]:
# Load dataset — latin1 encoding handles special characters in product names
df = pd.read_csv('../data/superstore.csv', encoding='latin1')
print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

---
<a id='inspect'></a>
## 4. Data Inspection

In [ ]:
print('=== Shape ===')
print(f'  {df.shape[0]:,} rows × {df.shape[1]} columns')
print('\n=== Data Types ===')
print(df.dtypes)
print('\n=== Numerical Statistics ===')
df.describe().round(3)

In [ ]:
print('=== Categorical Columns ===')
df.describe(include='object')

---
<a id='clean'></a>
## 5. Data Cleaning & Quality Checks

In [ ]:
# Missing values
missing = df.isnull().sum()
print('Missing values:', 'None' if missing.sum() == 0 else missing[missing > 0])

# Duplicate rows
dupes = df.duplicated().sum()
print(f'Duplicate rows: {dupes}')

# Convert dates
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%m/%d/%Y')
df['Ship Date']  = pd.to_datetime(df['Ship Date'],  format='%m/%d/%Y')

# Date ranges
print(f'Order Date range: {df["Order Date"].min().date()} → {df["Order Date"].max().date()}')
print(f'Ship Date range : {df["Ship Date"].min().date()} → {df["Ship Date"].max().date()}')

# Sanity checks
bad_ship = (df['Ship Date'] < df['Order Date']).sum()
print(f'Ship Date < Order Date (invalid): {bad_ship}')
print(f'Negative Sales: {(df["Sales"] < 0).sum()}')
print(f'Invalid Discounts: {((df["Discount"] < 0) | (df["Discount"] > 1)).sum()}')
print(f'Loss transactions (Profit ≤ 0): {(df["Profit"] <= 0).sum()} '
      f'({(df["Profit"] <= 0).mean()*100:.1f}%)')

In [ ]:
# Temporal feature engineering
df['Order Year']     = df['Order Date'].dt.year
df['Order Month']    = df['Order Date'].dt.month
df['Order Quarter']  = df['Order Date'].dt.quarter
df['Shipping Delay'] = (df['Ship Date'] - df['Order Date']).dt.days
df['Revenue_Per_Unit']= df['Sales'] / df['Quantity']
df['Is_Profitable']  = (df['Profit'] > 0).astype(int)

print('Features added. Ready for EDA.')
print(f'Shipping Delay: {df["Shipping Delay"].min()}–{df["Shipping Delay"].max()} days (mean={df["Shipping Delay"].mean():.1f})')

---
<a id='eda'></a>
## 6. Exploratory Data Analysis

In [ ]:
# Sales and Profit distributions
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(df['Sales'], bins=80, color='steelblue', edgecolor='white', log=True)
axes[0].set_title('Sales Distribution (log scale)'); axes[0].set_xlabel('Sales ($)')
axes[1].hist(df['Profit'], bins=80, color='coral', edgecolor='white')
axes[1].axvline(0, color='black', linestyle='--', label='Break-even')
axes[1].set_title('Profit Distribution'); axes[1].set_xlabel('Profit ($)'); axes[1].legend()
plt.suptitle('Sales & Profit Distributions', y=1.02)
plt.tight_layout()
plt.savefig(f'{CHARTS_DIR}/nb_01_distributions.png', bbox_inches='tight')
plt.show()

In [ ]:
# Monthly trend
monthly = (df.groupby(df['Order Date'].dt.to_period('M'))
             .agg(Sales=('Sales','sum'), Profit=('Profit','sum'))
             .reset_index())
monthly['Order Date'] = monthly['Order Date'].dt.to_timestamp()

fig, ax1 = plt.subplots(figsize=(13, 4))
ax1.fill_between(monthly['Order Date'], monthly['Sales'], alpha=0.25, color='steelblue')
ax1.plot(monthly['Order Date'], monthly['Sales'], color='steelblue', linewidth=1.5, label='Sales')
ax1.set_ylabel('Monthly Sales ($)', color='steelblue')
ax2 = ax1.twinx()
ax2.plot(monthly['Order Date'], monthly['Profit'], color='darkorange', linewidth=2, label='Profit')
ax2.axhline(0, color='red', linestyle='--', linewidth=0.8)
ax2.set_ylabel('Monthly Profit ($)', color='darkorange')
plt.title('Monthly Sales & Profit Trend (2014–2017)')
plt.tight_layout(); plt.savefig(f'{CHARTS_DIR}/nb_02_monthly_trend.png', bbox_inches='tight')
plt.show()

In [ ]:
# Category & Sub-Category
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cat = df.groupby('Category').agg(Sales=('Sales','sum'), Profit=('Profit','sum')).reset_index()
cat['Margin'] = (cat['Profit']/cat['Sales']*100).round(2)
axes[0].bar(cat['Category'], cat['Profit'],
            color=['steelblue','darkorange','seagreen'], edgecolor='white')
axes[0].set_title('Total Profit by Category')
axes[0].set_ylabel('Profit ($)')

subcat_p = (df.groupby('Sub-Category')['Profit'].sum().sort_values())
colors_p = ['coral' if v < 0 else 'steelblue' for v in subcat_p]
axes[1].barh(subcat_p.index, subcat_p.values, color=colors_p, edgecolor='white')
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Profit by Sub-Category (red = loss)')
axes[1].set_xlabel('Total Profit ($)')

plt.tight_layout(); plt.savefig(f'{CHARTS_DIR}/nb_03_category.png', bbox_inches='tight')
plt.show()

In [ ]:
# Region and Segment analysis
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

region = df.groupby('Region').agg(Sales=('Sales','sum'), Profit=('Profit','sum')).reset_index()
region['Margin'] = (region['Profit']/region['Sales']*100).round(2)
axes[0].bar(region.sort_values('Profit', ascending=False)['Region'],
            region.sort_values('Profit', ascending=False)['Profit'],
            color='steelblue', edgecolor='white')
axes[0].set_title('Profit by Region'); axes[0].set_ylabel('Profit ($)')

segment = df.groupby('Segment').agg(Sales=('Sales','sum'), Profit=('Profit','sum')).reset_index()
segment['Margin'] = (segment['Profit']/segment['Sales']*100).round(2)
axes[1].bar(segment.sort_values('Profit', ascending=False)['Segment'],
            segment.sort_values('Profit', ascending=False)['Profit'],
            color=['#3b82f6','#10b981','#f59e0b'], edgecolor='white')
axes[1].set_title('Profit by Segment'); axes[1].set_ylabel('Profit ($)')

plt.tight_layout(); plt.savefig(f'{CHARTS_DIR}/nb_04_region_segment.png', bbox_inches='tight')
plt.show()

print('Region summary:')
print(region[['Region','Sales','Profit','Margin']].sort_values('Profit', ascending=False).to_string(index=False))
print('\nSegment summary:')
print(segment[['Segment','Sales','Profit','Margin']].sort_values('Profit', ascending=False).to_string(index=False))

In [ ]:
# Discount vs Profit
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sample = df.sample(3000, random_state=RANDOM_STATE)
axes[0].scatter(sample['Discount'], sample['Profit'], alpha=0.2, s=10, color='steelblue')
axes[0].axhline(0, color='red', linestyle='--', linewidth=1)
axes[0].set_title('Discount vs Profit'); axes[0].set_xlabel('Discount'); axes[0].set_ylabel('Profit ($)')

disc_avg = df.groupby('Discount')['Profit'].mean().reset_index()
bar_colors = ['coral' if v < 0 else 'steelblue' for v in disc_avg['Profit']]
axes[1].bar(disc_avg['Discount'].astype(str), disc_avg['Profit'], color=bar_colors, edgecolor='white')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Avg Profit by Discount Level')
axes[1].set_xlabel('Discount'); axes[1].set_ylabel('Avg Profit ($)')
axes[1].tick_params(axis='x', rotation=45)

corr = df['Discount'].corr(df['Profit'])
print(f'Pearson correlation (Discount, Profit): {corr:.4f}')
print(f'Avg profit with 0% discount : ${df[df["Discount"]==0]["Profit"].mean():,.2f}')
print(f'Avg profit with >30% discount: ${df[df["Discount"]>0.3]["Profit"].mean():,.2f}')

plt.tight_layout(); plt.savefig(f'{CHARTS_DIR}/nb_05_discount.png', bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap
num_cols = ['Sales','Quantity','Discount','Profit','Shipping Delay','Order Month','Order Quarter']
corr_matrix = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            mask=mask, square=True, linewidths=0.5, ax=ax)
ax.set_title('Correlation Matrix — Numerical Features')
plt.tight_layout(); plt.savefig(f'{CHARTS_DIR}/nb_06_correlation.png', bbox_inches='tight')
plt.show()

---
<a id='kpi'></a>
## 7. Business KPIs & Analysis

In [ ]:
kpis = {
    'Total Sales':           df['Sales'].sum(),
    'Total Profit':          df['Profit'].sum(),
    'Total Orders':          df['Order ID'].nunique(),
    'Total Quantity Sold':   df['Quantity'].sum(),
    'Total Customers':       df['Customer ID'].nunique(),
    'Avg Order Value':       df.groupby('Order ID')['Sales'].sum().mean(),
    'Profit Margin (%)':     df['Profit'].sum() / df['Sales'].sum() * 100,
    'Avg Discount (%)':      df['Discount'].mean() * 100,
    'Loss Rate (%)':         (df['Profit'] <= 0).mean() * 100,
}

print('=== KEY PERFORMANCE INDICATORS ===')
for k, v in kpis.items():
    if 'Sales' in k or 'Profit' in k or 'Value' in k:
        print(f'  {k:<25} ${v:>14,.2f}')
    elif 'Orders' in k or 'Quantity' in k or 'Customers' in k:
        print(f'  {k:<25} {v:>14,.0f}')
    else:
        print(f'  {k:<25} {v:>14.2f}%')

In [ ]:
# Loss-making sub-categories
sc_summary = df.groupby('Sub-Category').agg(
    Total_Sales=('Sales','sum'), Total_Profit=('Profit','sum'), Transactions=('Row ID','count')
).assign(Margin=lambda x: (x['Total_Profit']/x['Total_Sales']*100).round(2))

loss_sc = sc_summary[sc_summary['Total_Profit'] < 0].sort_values('Total_Profit')
print('Loss-making sub-categories:')
print(loss_sc[['Total_Sales','Total_Profit','Margin','Transactions']].to_string())

In [ ]:
# Top / bottom products
top5_s  = df.groupby('Product Name')['Sales'].sum().nlargest(5)
top5_p  = df.groupby('Product Name')['Profit'].sum().nlargest(5)
bot5_p  = df.groupby('Product Name')['Profit'].sum().nsmallest(5)

print('TOP 5 by Sales:')
for n, v in top5_s.items(): print(f'  ${v:>9,.2f}  {n[:70]}')
print('\nTOP 5 by Profit:')
for n, v in top5_p.items(): print(f'  ${v:>9,.2f}  {n[:70]}')
print('\nBOTTOM 5 (Biggest Losses):')
for n, v in bot5_p.items(): print(f'  ${v:>9,.2f}  {n[:70]}')

---
<a id='features'></a>
## 8. Feature Engineering

**Leakage prevention:** `Profit` is never used as an input feature for either model.  
`Is_Profitable` (classification target) is only derived from `Profit` and excluded from inputs.

In [ ]:
CATEGORICAL_COLS = ['Ship Mode', 'Segment', 'Region', 'Category', 'Sub-Category']
NUMERIC_COLS     = ['Quantity', 'Discount', 'Order Month', 'Order Quarter', 'Order Year', 'Shipping Delay']

ml = df[CATEGORICAL_COLS + NUMERIC_COLS + ['Sales', 'Profit', 'Is_Profitable']].copy()
ml_enc = pd.get_dummies(ml, columns=CATEGORICAL_COLS, drop_first=False)

FEAT_COLS = [c for c in ml_enc.columns if c not in ['Sales','Profit','Is_Profitable']]

X      = ml_enc[FEAT_COLS]
y_sales = ml_enc['Sales']
y_class = ml_enc['Is_Profitable']

X_tr, X_te, ys_tr, ys_te, yc_tr, yc_te = train_test_split(
    X, y_sales, y_class, test_size=0.2, random_state=RANDOM_STATE, stratify=y_class
)

print(f'Feature matrix: {X.shape}')
print(f'Train: {X_tr.shape[0]:,}  |  Test: {X_te.shape[0]:,}')
print(f'Profitable rate — Train: {yc_tr.mean()*100:.1f}%  |  Test: {yc_te.mean()*100:.1f}%')

---
<a id='regression'></a>
## 9. Sales Prediction — Regression Models

In [ ]:
scaler = StandardScaler()
Xtr_sc = scaler.fit_transform(X_tr)
Xte_sc = scaler.transform(X_te)

def eval_reg(name, model, Xtr, Xte, ytr, yte):
    model.fit(Xtr, ytr)
    yp   = model.predict(Xte)
    mae  = mean_absolute_error(yte, yp)
    rmse = float(np.sqrt(mean_squared_error(yte, yp)))
    r2   = r2_score(yte, yp)
    print(f'  {name:<30}  MAE={mae:>8.2f}  RMSE={rmse:>9.2f}  R²={r2:.4f}')
    return dict(name=name, mae=round(mae,2), rmse=round(rmse,2), r2=round(r2,4), model=model, y_pred=yp)

reg_results = []
print('=== Regression Results (Test Set) ===')
reg_results.append(eval_reg('Linear Regression', LinearRegression(), Xtr_sc, Xte_sc, ys_tr, ys_te))
reg_results.append(eval_reg('Ridge Regression',  Ridge(alpha=10.0),  Xtr_sc, Xte_sc, ys_tr, ys_te))
reg_results.append(eval_reg('Random Forest',
    RandomForestRegressor(n_estimators=200, max_depth=12, random_state=RANDOM_STATE, n_jobs=-1),
    X_tr, X_te, ys_tr, ys_te))
reg_results.append(eval_reg('Gradient Boosting',
    GradientBoostingRegressor(n_estimators=200, learning_rate=0.08, max_depth=5,
                               subsample=0.8, random_state=RANDOM_STATE),
    X_tr, X_te, ys_tr, ys_te))

In [ ]:
best_reg = max(reg_results, key=lambda x: x['r2'])
print(f'Best regression model: {best_reg["name"]}  (R²={best_reg["r2"]})')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
lim = max(ys_te.max(), best_reg['y_pred'].max())
axes[0].scatter(ys_te, best_reg['y_pred'], alpha=0.2, s=10, color='steelblue')
axes[0].plot([0, lim], [0, lim], 'r--', linewidth=1.2, label='Perfect')
axes[0].set_title(f'Actual vs Predicted Sales\n({best_reg["name"]})')
axes[0].set_xlabel('Actual ($)'); axes[0].set_ylabel('Predicted ($)'); axes[0].legend()
residuals = ys_te.values - best_reg['y_pred']
axes[1].hist(residuals, bins=60, color='coral', edgecolor='white')
axes[1].axvline(0, color='black', linestyle='--')
axes[1].set_title('Residuals'); axes[1].set_xlabel('Residual ($)')
plt.tight_layout(); plt.savefig(f'{CHARTS_DIR}/nb_07_regression.png', bbox_inches='tight')
plt.show()

---
<a id='classification'></a>
## 10. Profitability Classification Models

**Target:** `Is_Profitable` — 1 if Profit > 0, else 0.  
**Anti-leakage:** Profit and all its direct derivatives are excluded from input features.

In [ ]:
def eval_cls(name, model, Xtr, Xte, ytr, yte):
    model.fit(Xtr, ytr)
    yp   = model.predict(Xte)
    yprb = model.predict_proba(Xte)[:,1]
    rep  = classification_report(yte, yp, output_dict=True, zero_division=0)
    auc  = roc_auc_score(yte, yprb)
    print(f'\n=== {name} ===')
    print(classification_report(yte, yp, target_names=['Loss/Break-even','Profitable'], zero_division=0))
    print(f'ROC-AUC: {auc:.4f}')
    pk = 1 if 1 in rep else '1'
    return dict(name=name, accuracy=round(rep['accuracy'],4),
                precision=round(rep[pk]['precision'],4),
                recall=round(rep[pk]['recall'],4),
                f1=round(rep[pk]['f1-score'],4),
                roc_auc=round(auc,4),
                model=model, y_pred=yp, y_proba=yprb)

Xtr_sc2 = scaler.fit_transform(X_tr)
Xte_sc2 = scaler.transform(X_te)

cls_results = []
cls_results.append(eval_cls('Logistic Regression',
    LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE),
    Xtr_sc2, Xte_sc2, yc_tr, yc_te))
cls_results.append(eval_cls('Random Forest',
    RandomForestClassifier(n_estimators=200, max_depth=12, class_weight='balanced',
                            random_state=RANDOM_STATE, n_jobs=-1),
    X_tr, X_te, yc_tr, yc_te))
cls_results.append(eval_cls('Gradient Boosting',
    GradientBoostingClassifier(n_estimators=200, learning_rate=0.08, max_depth=5,
                                subsample=0.8, random_state=RANDOM_STATE),
    X_tr, X_te, yc_tr, yc_te))

---
<a id='eval'></a>
## 11. Model Evaluation & Comparison

In [ ]:
# Regression comparison table
print('=== Regression Model Comparison ===')
reg_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ('model','y_pred')} for r in reg_results])
print(reg_df.to_string(index=False))

# Classification comparison table
print('\n=== Classification Model Comparison ===')
cls_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ('model','y_pred','y_proba')} for r in cls_results])
print(cls_df.to_string(index=False))

In [ ]:
# Best classifier confusion matrix
best_cls = max(cls_results, key=lambda x: x['roc_auc'])
print(f'Best classifier: {best_cls["name"]}  ROC-AUC={best_cls["roc_auc"]}')

from sklearn.metrics import roc_curve
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

cm = confusion_matrix(yc_te, best_cls['y_pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred Loss','Pred Profit'],
            yticklabels=['True Loss','True Profit'], ax=axes[0])
axes[0].set_title(f'Confusion Matrix — {best_cls["name"]}')

fpr, tpr, _ = roc_curve(yc_te, best_cls['y_proba'])
axes[1].plot(fpr, tpr, color='steelblue', linewidth=2, label=f'AUC={best_cls["roc_auc"]:.4f}')
axes[1].plot([0,1],[0,1],'k--',linewidth=0.8)
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].set_title('ROC Curve'); axes[1].legend()

plt.tight_layout(); plt.savefig(f'{CHARTS_DIR}/nb_08_classification.png', bbox_inches='tight')
plt.show()

---
<a id='importance'></a>
## 12. Feature Importance & Interpretability

> **Note on causation vs association:** Feature importance from tree-based models shows which features the model uses most for prediction. This reflects *association* in the training data, not necessarily *causation*. Discount is highly associated with losses, but the actual causal mechanism is complex (pricing, product type, customer negotiation).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Sales prediction — best tree model
best_tree_reg = next((r for r in reversed(reg_results) if hasattr(r['model'], 'feature_importances_')), None)
if best_tree_reg:
    imp_r = pd.Series(best_tree_reg['model'].feature_importances_, index=FEAT_COLS).nlargest(15).sort_values()
    imp_r.plot(kind='barh', ax=axes[0], color='steelblue', edgecolor='white')
    axes[0].set_title(f'Top 15 Features — Sales Prediction\n({best_tree_reg["name"]})')
    axes[0].set_xlabel('Importance')

# Profitability classification — best tree model
best_tree_cls = next((r for r in reversed(cls_results) if hasattr(r['model'], 'feature_importances_')), None)
if best_tree_cls:
    imp_c = pd.Series(best_tree_cls['model'].feature_importances_, index=FEAT_COLS).nlargest(15).sort_values()
    imp_c.plot(kind='barh', ax=axes[1], color='darkorange', edgecolor='white')
    axes[1].set_title(f'Top 15 Features — Profitability\n({best_tree_cls["name"]})')
    axes[1].set_xlabel('Importance')

plt.tight_layout(); plt.savefig(f'{CHARTS_DIR}/nb_09_importance.png', bbox_inches='tight')
plt.show()

if best_tree_cls:
    print('Top 5 features for Profitability Prediction (association, not causation):')
    for feat, val in imp_c.tail(5)[::-1].items():
        print(f'  {feat:<35} {val:.4f}')

---
<a id='insights'></a>
## 13. Business Insights & Recommendations

In [ ]:
print('=== ACTUAL BUSINESS FINDINGS ===')
print()
print(f'1. Overall profit margin: {df["Profit"].sum()/df["Sales"].sum()*100:.2f}%')
print(f'2. Loss transaction rate: {(df["Profit"]<=0).mean()*100:.1f}%')
print(f'3. Discount–Profit correlation: {df["Discount"].corr(df["Profit"]):.4f}')
print(f'4. Loss-making sub-categories: {list(sc_summary[sc_summary["Total_Profit"] < 0].index)}')
print(f'5. Most profitable region: {region.loc[region["Margin"].idxmax(), "Region"]} '
      f'({region["Margin"].max():.2f}% margin)')
print(f'6. Least profitable region: {region.loc[region["Margin"].idxmin(), "Region"]} '
      f'({region["Margin"].min():.2f}% margin)')
print(f'7. Best classification ROC-AUC: {best_cls["roc_auc"]} ({best_cls["name"]})')

### Recommendations

| # | Recommendation | Evidence |
|---|---|---|
| 1 | **Cap discounts at 20%** | Discounts ≥ 30% produce average losses; correlation = −0.22 |
| 2 | **Review Tables & Bookcases pricing** | Net losses of $17.7k and $3.5k respectively |
| 3 | **Deploy profitability classifier at order entry** | ROC-AUC = 0.98 — reliable early warning |
| 4 | **Grow West region presence** | Highest margin at 14.9% |
| 5 | **Prioritise Home Office & Corporate segments** | Both outperform Consumer margin |
| 6 | **Leverage Q4 demand surge** | Strongest months: Oct, Nov, Dec |
| 7 | **Push Copier/Printer category** | Canon Copier: $25k profit alone |
| 8 | **Discontinue Cubify 3D Printers** | $12.7k combined losses from 2 SKUs |

---
<a id='conclusion'></a>
## 14. Conclusion

This project delivered a complete end-to-end data analytics and AI pipeline on the Superstore dataset:

- **EDA** revealed that 19.4% of transactions are loss-making, primarily driven by discounting and specific sub-categories (Tables, Bookcases, Supplies).
- **Sales Prediction** (R² ≈ 0.28) provides a moderate baseline — sales variance is largely explained by product category/type, which limits linear predictability across all SKUs.
- **Profitability Classification** (ROC-AUC ≈ 0.98) is highly actionable — the model can reliably flag loss-making orders before they ship, enabling pricing intervention.
- **Discount** is overwhelmingly the strongest predictor of losses (importance > 0.76 in the tree classifier).

The trained models are saved and served through a FastAPI backend, with an interactive Streamlit dashboard for business users.